# 01 - Getting the data

**Question of this notebook:** how do the lap times of a race get from the
internet into one clean table?

Source: [TracingInsights](https://github.com/TracingInsights), one GitHub
repository per season. For every race and session there is one folder, and
for every driver one file `laptimes.json` with one entry per lap.

Everything here is done by the functions of `boxbox/data.py`. This notebook
shows what each one returns, in the order the code uses them:

`list_races` -> `find_race` -> `list_drivers` -> `load_driver_laps` ->
`load_laps` -> `get_laps` (with the CSV snapshots) -> `prepare_laps`

**Internet:** sections 1 to 4 download from GitHub (about 5 seconds for a whole
race). From section 5 on, the race is read from the CSV snapshot on disk.

**Warning:** never call a variable `data` in a notebook. It hides the module
and gives errors like `'dict' object has no attribute 'get_laps'`.

In [1]:
import os
import sys

# The notebooks live in notebooks/, the package in the project root.
sys.path.append(os.path.abspath(os.path.join("..")))

# Reload boxbox automatically when a .py file changes.
%load_ext autoreload
%autoreload 2

In [2]:
import time

from boxbox.data import (
    DataLoadError,
    find_race,
    get_laps,
    get_session_folder,
    list_drivers,
    list_races,
    load_driver_laps,
    load_laps,
    prepare_laps,
    snapshot_path,
)

## 1. Which races exist, and how are they found?

`list_races` asks GitHub for the folders of a season and keeps those that end
in "Grand Prix". `find_race` turns the text typed by the user into the exact
folder name: it ignores case and accepts a part of the name.

In [3]:
races = list_races(2022)
print(len(races), "races in 2022")
races[:5]

22 races in 2022


['Abu Dhabi Grand Prix',
 'Australian Grand Prix',
 'Austrian Grand Prix',
 'Azerbaijan Grand Prix',
 'Bahrain Grand Prix']

In [4]:
print(find_race(2022, "bahr"))
print(find_race(2022, "MONACO"))

Bahrain Grand Prix
Monaco Grand Prix


A wrong or too short name gives a clear `ValueError`, with a suggestion when
a name is close.

In [5]:
for text in ["bahrian", "grand"]:
    try:
        find_race(2022, text)
    except ValueError as error:
        print(str(error)[:160])

No 2022 race matches 'bahrian'. Did you mean: ['bahrain', 'austrian', 'hungarian']?
'grand' matches 22 races in 2022: ['Abu Dhabi Grand Prix', 'Australian Grand Prix', 'Austrian Grand Prix', 'Azerbaijan Grand Prix', 'Bahrain Grand Prix', 'Belgi


## 2. Sessions and drivers

A session code is translated into the folder name: `"R"` is `Race`, `"Q"` is
`Qualifying`, `"P1"` is `Practice 1`... An unknown code is refused before any
download.

In [6]:
print(get_session_folder("Q"))

try:
    get_session_folder("X")
except ValueError as error:
    print(error)

Qualifying
Invalid session 'X'. Use one of: R, Q, S, P1, P2, P3.


In [7]:
drivers = list_drivers(2022, "Bahrain Grand Prix", "R")
print(len(drivers), drivers)

20 ['VER', 'GAS', 'PER', 'ALO', 'LEC', 'STR', 'MAG', 'TSU', 'ALB', 'ZHO', 'HUL', 'RIC', 'OCO', 'NOR', 'HAM', 'MSC', 'SAI', 'LAT', 'RUS', 'BOT']


## 3. The laps of one driver

One row per lap, one column per field of `laptimes.json` (42 columns).
**Missing values are the text `"None"`**, not a real empty value: look at
`pin` and `pout` (the pit entry and exit), which are `"None"` on a normal lap.

In [8]:
lec = load_driver_laps(2022, "Bahrain Grand Prix", "R", "LEC")
print(lec.shape)
lec[["lap", "time", "compound", "stint", "pin", "pout"]].head()

(57, 42)


,lap,time,compound,stint,pin,pout
0,1,99.070,SOFT,1,None,None
1,2,97.853,SOFT,1,None,None
2,3,98.272,SOFT,1,None,None
3,4,98.414,SOFT,1,None,None
4,5,98.471,SOFT,1,None,None


A wrong driver code has no file, so the download fails with a `DataLoadError`
(the message says which address could not be downloaded).

In [9]:
try:
    load_driver_laps(2022, "Bahrain Grand Prix", "R", "XXX")
except DataLoadError as error:
    print(str(error)[:120])

Could not download https://raw.githubusercontent.com/TracingInsights/2022/main/Bahrain Grand Prix/Race/XXX/laptimes.json


## 4. The whole race

`load_laps` finds the race, lists the drivers, downloads the 20 files and
stacks them. It stops with a `DataLoadError` if one download fails, so a
table is never returned with a driver missing. The result is in blocks: all
the laps of the first driver (in the order of `drivers.json`), then the next.
It is **not** sorted by lap.

In [10]:
start = time.time()
raw = load_laps(2022, "bahrain")
print(f"{time.time() - start:.1f} seconds")
print(raw.shape, raw["drv"].nunique(), "drivers")

4.8 seconds
(1125, 42) 20 drivers


In [11]:
# The end of the first block and the start of the second.
raw[["drv", "lap"]].iloc[50:58]

,drv,lap
50,VER,51
51,VER,52
52,VER,53
53,VER,54
54,GAS,1
55,GAS,2
56,GAS,3
57,GAS,4


## 5. Snapshots: download once, read from disk afterwards

Twenty downloads on every call is not acceptable. `get_laps` saves the raw
table as a CSV in `data/snapshots/` the first time and reads that file
afterwards. A new download is made only if the file is missing or
`refresh=True`.

The file name is built from the text typed by the user, so that it can be
found **without the internet**. The limit: two spellings of the same race
give two files.

In [12]:
print(snapshot_path(2022, "bahrain", "R").name)
print(snapshot_path(2022, "Bahrain", "r").name)  # same file
print(snapshot_path(2022, "Bahrain Grand Prix", "R").name)  # another file

2022_bahrain_R.csv
2022_bahrain_R.csv
2022_bahrain_grand_prix_R.csv


In [13]:
start = time.time()
laps_csv = get_laps(2022, "bahrain")  # downloads only if the file is missing
print(f"first call:  {time.time() - start:.2f} seconds")
# On a fresh clone without the snapshot, this first call takes about 5 seconds.

start = time.time()
laps_csv = get_laps(2022, "bahrain")
print(f"second call: {time.time() - start:.2f} seconds (read from disk)")

first call:  0.00 seconds
second call: 0.00 seconds (read from disk)


The table always comes from the CSV, so it is the same whether or not a
download happened. Two consequences, both tested:

- `"None"` becomes a real missing value (`NaN`) when the CSV is read;
- `status` is read as **text** on purpose: as a number, `"12"` would be the
  number 12 and a comparison with `"1"` would never be true.

In [14]:
print(repr(raw["pin"].iloc[0]), "->", repr(laps_csv["pin"].iloc[0]))
print(laps_csv["status"].dtype, laps_csv["status"].unique()[:5])

'None' -> np.float64(nan)
str <StringArray>
['1', '12', '164', '4', '41']
Length: 5, dtype: str


## 6. Types and columns: `prepare_laps`

The table has 42 columns, and the analysis needs 14. `prepare_laps` keeps
those and turns `"None"` into `NaN` in four columns, so they become numbers:
`time`, `pos`, `pin`, `pout`. The other columns already have the right type
(a numeric column becomes `object` exactly when it contains the text
`"None"`). The input table is not modified.

| Column | Meaning | Used for |
|---|---|---|
| `drv`, `team` | driver code, team name | who |
| `lap` | lap number | x-axis |
| `time` | lap time in seconds | pace, degradation |
| `compound` | SOFT / MEDIUM / HARD / ... | strategy, degradation |
| `stint` | stint number (+1 after each pit stop) | strategy |
| `life` | laps already driven on this set of tyres | degradation |
| `fresh` | new tyre or used one | kept, but not used: it equals `life == 1` at the start of a stint |
| `pos` | position at the end of the lap | race story |
| `status` | track status during the lap | cleaning |
| `pin`, `pout` | pit entry and exit time | cleaning |
| `del`, `iacc` | lap deleted, timing accurate | cleaning |

In [15]:
prepared = prepare_laps(laps_csv)
print(laps_csv.shape, "->", prepared.shape)
prepared.dtypes

(1125, 42) -> (1125, 14)


time        float64
lap           int64
compound        str
stint         int64
life          int64
pos         float64
status          str
drv             str
pout        float64
pin         float64
fresh          bool
team            str
del            bool
iacc           bool
dtype: object

In [16]:
# Missing values per column: pin and pout are mostly missing (only the
# 59 in-laps and 58 out-laps have a value).
prepared.isna().sum()

time           2
lap            0
compound       0
stint          0
life           0
pos            2
status         0
drv            0
pout        1067
pin         1066
fresh          0
team           0
del            0
iacc           0
dtype: int64

In [17]:
# The two laps without a time: the last laps of GAS (lap 45) and PER (lap 57).
prepared[prepared["time"].isna()]

,time,lap,compound,stint,life,pos,status,drv,pout,pin,fresh,team,del,iacc
98,NaN,45,HARD,3,13,NaN,164,GAS,NaN,NaN,True,AlphaTauri,False,False
155,NaN,57,SOFT,4,17,NaN,12,PER,NaN,NaN,False,Red Bull Racing,False,False


## Summary

- `get_laps(year, race)` is the function the other notebooks use: it returns
  the raw table of the race, from the CSV snapshot (downloaded once).
- `prepare_laps` keeps the 14 columns the analysis needs and turns the text
  `"None"` into real missing values in `time`, `pos`, `pin` and `pout`.
- The prepared Bahrain 2022 table has 1125 laps (20 drivers). Two laps have no
  time: the last laps of GAS and PER.
- Next: `02_cleaning.ipynb` decides which of these laps are representative.